%share of AI roles with perk X = %share of perk X in general + %share of AI roles + log(size of occupation-year) + log(salary of AI roles)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
from IPython.core.display import HTML

In [ ]:
import numpy as np

In [ ]:
# import stargazer
from stargazer.stargazer import Stargazer

In [ ]:
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)

In [ ]:
from package_files.benefits_defns import *

In [ ]:
import importlib
import package_files.benefits_defns


In [ ]:
importlib.reload(package_files.benefits_defns)


In [ ]:
df = pd.read_csv('../exports/occ_year_coeff_analysis.csv')

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df[(df['Occupation']=="Architecture and Engineering Occupations") & (df['Year']==2019)]

In [ ]:
df

In [ ]:
# add column for log job count
df['Log Job Count'] = df['job_count'].apply(lambda x: np.log(x))

%share of AI roles with perk X = %share of perk X in general + %share of AI roles + log(size of occupation-year) + log(salary of AI roles)

# Run Models

In [ ]:

# benefit_models = {}
benefit_models = []
for benefit in benefits4:
    benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)

    label = benefits_labels_map[benefit]

    X = benefit_df[[f'Overall Benefit Prevalence', 'AI Demand', 'Log Job Count', 'Mean Log AI Salary']]
    y = benefit_df[f'Prevalence: {label} (AI)']

    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    # benefit_models.setdefault(benefit, []).append(model)
    benefit_models.append(model)
    # Print the summary of the regression
    print(model.summary())

# Display Results

In [ ]:
stargazer = Stargazer(benefit_models)
display(HTML(stargazer.render_html()))

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(benefit_models)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column

with open(f'../exports/occ_year_final_model.tex', 'w') as f:
    f.write(stargazer.render_latex())
# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


In [ ]:
# for benefit in benefits4:
#     stargazer = Stargazer(benefit_models[benefit])
#     stargazer.title(f'Coefficients for {benefits_labels_map[benefit]}')
#     display(HTML(stargazer.render_html()))
#     # # export to latex
#     # with open(f'../exports/occ_year_coeff_{benefit}.tex', 'w') as f:
#     #     f.write(stargazer.render_latex())

# Remote Keywords

In [ ]:
df = pd.read_csv('../exports/occ_year_data/occ_year_analysis_remotekw.csv')

In [ ]:
df

In [ ]:
benefit_models = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df = benefit_df[benefit_df[f'Prevalence: {benefits_labels_map[benefit]} (AI)'].notna()]
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)

    label = benefits_labels_map[benefit]

    X = benefit_df[[f'Overall Benefit Prevalence', 'AI Demand', 'Log Job Count', 'MEDIAN SALARY_ai']]
    y = benefit_df[f'Prevalence: {label} (AI)']

    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    # benefit_models.setdefault(benefit, []).append(model)
    benefit_models.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
benefit_models

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(benefit_models)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


# Model Configuration

## Baseline: Year and Overall Benefit Prevalence

In [ ]:
baseline_models = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df = benefit_df[benefit_df[f'Prevalence: {benefits_labels_map[benefit]} (AI)'].notna()]
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]

    X = benefit_df[[f'Overall Benefit Prevalence']]
    y = benefit_df[f'Prevalence: {label} (AI)']
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    # benefit_models.setdefault(benefit, []).append(model)
    baseline_models.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(baseline_models)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


## With AI Demand

In [ ]:
models2 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df = benefit_df[benefit_df[f'Prevalence: {benefits_labels_map[benefit]} (AI)'].notna()]
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]

    X = benefit_df[[f'Overall Benefit Prevalence', 'AI Demand']]
    y = benefit_df[f'Prevalence: {label} (AI)']
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    # benefit_models.setdefault(benefit, []).append(model)
    models2.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
models2

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models2)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


## + % Demand Change

In [ ]:
models3 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    X_cols = [f'Overall Benefit Prevalence', 'AI Demand', 'AI Demand % Change']
    y_col = [f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    # benefit_models.setdefault(benefit, []).append(model)
    models3.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
models3

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models3)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


## % Change Substitute for AI Demand

In [ ]:
models3b = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    X_cols = [f'Overall Benefit Prevalence', 'AI Demand % Change']
    y_col = [f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    # benefit_models.setdefault(benefit, []).append(model)
    models3b.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models3b)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


## Median Salary

In [ ]:
models4 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    X_cols = [f'Overall Benefit Prevalence', 'AI Demand', 'MEDIAN_SALARY_ai']
    y_col = [f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    # benefit_models.setdefault(benefit, []).append(model)
    models4.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models4)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


In [ ]:
models_salary = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    X_cols = [f'MEDIAN_LOG_SALARY_ai']
    y_col = [f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    # X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    # benefit_models.setdefault(benefit, []).append(model)
    models_salary.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models_salary)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


In [ ]:
df['MEDIAN SALARY_ai'].describe()

In [ ]:
df['MEDIAN SALARY_ai'].hist()

In [ ]:
df['MEDIAN SALARY_non_ai'].hist()

In [ ]:
df['MEDIAN_LOG_SALARY_ai'].hist()

In [ ]:
df['MEDIAN_LOG_SALARY_non_ai'].hist()

# Computer & Mathematical Occupations Dummy

In [ ]:
df['Comp_Math_Occ'] = (df[occupation] == 'Computer and Mathematical Occupations').astype(int)

In [ ]:
models5 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence', 'LOG_SALARY_ai':'Mean Log AI Salary'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    X_cols = [f'Overall Benefit Prevalence', 'AI Demand', 'Comp_Math_Occ']
    y_col = [f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    # benefit_models.setdefault(benefit, []).append(model)
    models5.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models5)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


# 2024

In [ ]:
df_24 = 

# Raw Numbers

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../exports/occ_year_data/occ_year_analysis_2024_raw.csv')

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df

In [ ]:
# df[df[occupation]=='Computer and Mathematical Occupations']

In [ ]:
models4 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    X_cols = [f'Overall Benefit Prevalence', 'ai_role_count', 'MEDIAN_LOG_SALARY_ai', 'job_count']
    y_col = [f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    models4.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models4)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


## No Year Fixed Effects

In [ ]:
models4 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    X_cols = [f'Overall Benefit Prevalence', 'ai_role_count', 'MEDIAN_LOG_SALARY_ai', 'job_count']
    y_col = [f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    # year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    # X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)

    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    models4.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models4)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))


## Log

In [ ]:
models4 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence', 'ai_role_count', f'Prevalence: {label} (AI)']
    # get log of X_cols
    for col in log_cols:
        benefit_df[f'{col} (Log)'] = benefit_df[col].apply(lambda x: np.log(x +1))
    X_cols = [f'{col} (Log)' for col in log_cols[:-1]] + ['MEDIAN_LOG_SALARY_ai', 'Log Job Count'] 
    y_col = [f'Prevalence: {label} (AI) (Log)']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    models4.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models4)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))

## Log No Year

In [ ]:
models4 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence', 'ai_role_count', f'Prevalence: {label} (AI)']
    # get log of X_cols
    for col in log_cols:
        benefit_df[f'{col} (Log)'] = benefit_df[col].apply(lambda x: np.log(x +1))
    X_cols = [f'{col} (Log)' for col in log_cols[:-1]] + ['MEDIAN_LOG_SALARY_ai', 'Log Job Count'] 
    y_col = [f'Prevalence: {label} (AI) (Log)']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    # year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    # X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    models4.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import display, HTML

# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models4)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))

In [ ]:
df['Log AI Job Change'] = df['AI Job Count Change'].apply(lambda x: np.log(x +1))

In [ ]:
df['AI Job Count Change'].describe()

In [ ]:
df['Log AI Job Change'].value_counts(dropna=False)

## Change in Demand

In [ ]:
models4 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence', f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=log_cols)
    # get log of X_cols
    for col in log_cols:
        benefit_df[f'{col} (Log)'] = benefit_df[col].apply(lambda x: np.log(x +1))
    X_cols = [f'{col} (Log)' for col in log_cols[:-1]] + ['MEDIAN_LOG_SALARY_ai', 'Log Job Count', 'AI Job Count Change'] 
    y_col = [f'Prevalence: {label} (AI) (Log)']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    models4.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models4)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))

### Log Change

In [ ]:
models4 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence', 'AI Job Count Change','ai_role_count', f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=log_cols)
    # get log of X_cols
    for col in log_cols:
        benefit_df[f'{col} (Log)'] = benefit_df[col].apply(lambda x: np.sign(x) * np.log(abs(x) + 1))
    X_cols = [f'{col} (Log)' for col in log_cols[:-1]] + ['MEDIAN_LOG_SALARY_ai', 'Log Job Count'] 
    y_col = [f'Prevalence: {label} (AI) (Log)']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    models4.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models4)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column

# export to latex table
with open(f'../exports/tables/occ_year_model_log_raw.tex', 'w') as f:
    f.write(stargazer.render_latex())


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))

### Without Years

In [ ]:
models4 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence', 'AI Job Count Change', 'ai_role_count', f'Prevalence: {label} (AI)']
    benefit_df = benefit_df.dropna(subset=log_cols)
    # get log of X_cols
    for col in log_cols:
        benefit_df[f'{col} (Log)'] = benefit_df[col].apply(lambda x: np.sign(x) * np.log(abs(x) + 1))
    X_cols = [f'{col} (Log)' for col in log_cols[:-1]] + ['MEDIAN_LOG_SALARY_ai', 'Log Job Count'] 
    y_col = [f'Prevalence: {label} (AI) (Log)']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    # year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    # X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    # print(X)
    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    models4.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(models4)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))

# Differences

In [ ]:
df

In [ ]:
diff_models = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns = {f'ai_role_count': 'AI Job Count', 'MEDIAN_LOG_SALARY_ai': 'Median Log Salary AI'}, inplace=True)
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence']
    benefit_df = benefit_df.dropna(subset=log_cols)
    # get log of X_cols
    for col in log_cols:
        benefit_df[f'Log {col}'] = benefit_df[col].apply(lambda x: np.sign(x) * np.log(abs(x) + 1))
    X_cols = [f'Log {col}' for col in log_cols] + ['Median Log Salary AI'] 
    y_col = [f'{benefit}_difference']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    X = sm.add_constant(X)

    # Fit model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    diff_models.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
diff_models_2 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns = {f'ai_role_count': 'AI Job Count', 'MEDIAN_LOG_SALARY_ai': 'Median Log Salary AI'}, inplace=True)
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence', 'AI Job Count']
    benefit_df = benefit_df.dropna(subset=log_cols)
    # get log of X_cols
    for col in log_cols:
        benefit_df[f'Log {col}'] = benefit_df[col].apply(lambda x: np.sign(x) * np.log(abs(x) + 1))
    X_cols = [f'Log {col}' for col in log_cols] + ['Median Log Salary AI'] 
    y_col = [f'{benefit}_difference']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    X = sm.add_constant(X)

    # Fit model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    diff_models_2.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
combined_models = []
for model_a, model_b in zip(diff_models, diff_models_2):
    combined_models.extend([model_a, model_b])

# Create the Stargazer table
stargazer = Stargazer(combined_models)

# Set up the custom column labels for the top row (spanning two models per benefit)
benefit_labels = []
model_counts = []
for label in benefits4_labels:
    benefit_labels.append(label)
    model_counts.append(2)  # Each benefit spans two columns

# Apply custom columns for the top row (benefits)
stargazer.custom_columns(benefit_labels, model_counts)

# Add the second row for "A" and "B"
# stargazer.add_line(["A", "B"] * len(benefits4_labels))

# Render the table
display(HTML(stargazer.render_html()))

html_table = stargazer.render_html()  # Render HTML for Word

with open("../exports/tables/difference_regression_table_combined_2.html", "w") as f:
    f.write(html_table)

# Balanced Sample Differences

In [ ]:
diffs = pd.read_csv('../exports/occ_year_data/balanced_sample_diffs.csv')

In [ ]:
diffs

In [ ]:
df

In [ ]:
df = df.merge(diffs, on=[occupation, 'YEAR'])

In [ ]:
diff_models = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns = {f'ai_role_count': 'AI Job Count', 'MEDIAN_LOG_SALARY_ai': 'Median Log Salary AI'}, inplace=True)
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence']
    benefit_df = benefit_df.dropna(subset=log_cols)
    # get log of X_cols
    for col in log_cols:
        benefit_df[f'Log {col}'] = benefit_df[col].apply(lambda x: np.sign(x) * np.log(abs(x) + 1))
    X_cols = [f'Log {col}' for col in log_cols] + ['Median Log Salary AI', 'Log Job Count'] 
    y_col = [f'{benefit}_difference']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    X = sm.add_constant(X)

    # Fit model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    diff_models.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(diff_models)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` specifies that each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))

In [ ]:
html_table = stargazer.render_html()  # Render HTML for Word

with open("../exports/tables/difference_regression_table.html", "w") as f:
    f.write(html_table)

In [ ]:
diff_models_2 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns = {f'ai_role_count': 'AI Job Count', 'MEDIAN_LOG_SALARY_ai': 'Median Log Salary AI'}, inplace=True)
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence', 'AI Job Count']
    benefit_df = benefit_df.dropna(subset=log_cols)
    # get log of X_cols
    for col in log_cols:
        benefit_df[f'Log {col}'] = benefit_df[col].apply(lambda x: np.sign(x) * np.log(abs(x) + 1))
    X_cols = [f'Log {col}' for col in log_cols] + ['Median Log Salary AI', 'Log Job Count'] 
    y_col = [f'{benefit}_difference']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    X = sm.add_constant(X)

    # Fit model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    diff_models_2.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
[1]*len(benefits4)

In [ ]:
# Assume benefit_models is a list of models aligned with benefits4
stargazer = Stargazer(diff_models_2)

# Set custom column labels based on benefit names
stargazer.custom_columns(benefits4_labels, [1] * len(benefits4))  # `[1] * len(benefits4)` : each label is for one column


# Render the HTML display with benefit names as model titles
display(HTML(stargazer.render_html()))

In [ ]:
html_table = stargazer.render_html()  # Render HTML for Word

with open("../exports/tables/difference_regression_table_2.html", "w") as f:
    f.write(html_table)

In [ ]:
benefit_labels

In [ ]:
combined_models = []
for model_a, model_b in zip(diff_models, diff_models_2):
    combined_models.extend([model_a, model_b])

# Create the Stargazer table
stargazer = Stargazer(combined_models)

# Set up the custom column labels for the top row (spanning two models per benefit)
benefit_labels = []
model_counts = []
for label in benefits4_labels:
    benefit_labels.append(label)
    model_counts.append(2)  # Each benefit spans two columns

# Apply custom columns for the top row (benefits)
stargazer.custom_columns(benefit_labels, model_counts)

# Add the second row for "A" and "B"
# stargazer.add_line(["A", "B"] * len(benefits4_labels))

# Render the table
display(HTML(stargazer.render_html()))

In [ ]:
html_table = stargazer.render_html()  # Render HTML for Word

with open("../exports/tables/difference_regression_table_combined.html", "w") as f:
    f.write(html_table)

## Absolute Numbers

In [ ]:
diff_models_abs = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns = {f'ai_role_count': 'AI Job Count', 'MEDIAN_LOG_SALARY_ai': 'Median Log Salary AI'}, inplace=True)
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence']
    benefit_df = benefit_df.dropna(subset=log_cols)
    # get log of X_cols
    # for col in log_cols:
    #     benefit_df[f'Log {col}'] = benefit_df[col].apply(lambda x: np.sign(x) * np.log(abs(x) + 1))
    X_cols = ['Overall Benefit Prevalence','Median Log Salary AI', 'Log Job Count'] 
    y_col = [f'{benefit}_difference']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    X = sm.add_constant(X)

    # Fit model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    diff_models_abs.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
diff_models_abs_2 = []
for benefit in benefits4:
    # benefit_df = df[df['Benefit'] == benefit].copy()
    benefit_df = df.copy()
    benefit_df.rename(columns = {f'ai_role_count': 'AI Job Count', 'MEDIAN_LOG_SALARY_ai': 'Median Log Salary AI'}, inplace=True)
    benefit_df.rename(columns={f'Prevalence: {benefits_labels_map[benefit]}': f'Overall Benefit Prevalence'}, inplace=True)
    if benefit == 'wfh_wham':
        benefit_df = benefit_df[benefit_df['YEAR'] != 2018]
    label = benefits_labels_map[benefit]
    log_cols = [f'Overall Benefit Prevalence']
    benefit_df = benefit_df.dropna(subset=log_cols)
    # get log of X_cols
    # for col in log_cols:
    #     benefit_df[f'Log {col}'] = benefit_df[col].apply(lambda x: np.sign(x) * np.log(abs(x) + 1))
    X_cols = ['Overall Benefit Prevalence','Median Log Salary AI', 'Log Job Count', 'AI Job Count'] 
    y_col = [f'{benefit}_difference']
    
    benefit_df = benefit_df.dropna(subset=X_cols + y_col)
    X = benefit_df[X_cols]
    y = benefit_df[y_col]
    year_dummies = pd.get_dummies(benefit_df['YEAR'], prefix = 'Year').drop(columns=['Year_2019'])
    # print(year_dummies)
    X = pd.concat([X, year_dummies], axis=1)
    X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})
    X = sm.add_constant(X)

    # Fit model
    model = sm.OLS(y, X).fit()
    print(benefit)
    print("VIF Results")
    vif_data = []
    for i in range(X.shape[1]):
        vif = variance_inflation_factor(X.values, i)
        vif_data.append(vif)
    vif = pd.DataFrame()
    vif["features"] = X.columns
    vif["VIF"] = vif_data
    print(vif)
    # benefit_models.setdefault(benefit, []).append(model)
    diff_models_abs_2.append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
combined_models = []
for model_a, model_b in zip(diff_models_abs, diff_models_abs_2):
    combined_models.extend([model_a, model_b])

# Create the Stargazer table
stargazer = Stargazer(combined_models)

# Set up the custom column labels for the top row (spanning two models per benefit)
benefit_labels = []
model_counts = []
for label in benefits4_labels:
    benefit_labels.append(label)
    model_counts.append(2)  # Each benefit spans two columns

# Apply custom columns for the top row (benefits)
stargazer.custom_columns(benefit_labels, model_counts)

# Add the second row for "A" and "B"
# stargazer.add_line(["A", "B"] * len(benefits4_labels))

# Render the table
display(HTML(stargazer.render_html()))

In [ ]:
html_table = stargazer.render_html()  # Render HTML for Word

with open("../exports/tables/difference_regression_table_combined_absolute.html", "w") as f:
    f.write(html_table)

# Correlations

In [ ]:
# select all columns beginning with Prevalence: 
prevalence_cols = [col for col in df.columns if col.startswith('Prevalence: ')]
df_prevalence = df[prevalence_cols + ['MEAN SALARY']]

In [ ]:
df_prevalence

In [ ]:
correlation_matrix = df_prevalence.corr()

In [ ]:
correlation_matrix

In [ ]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    # Define outlier boundaries
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

In [ ]:
import seaborn as sns
plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, cmap='Blues', cbar = True)
plt.xticks(ticks=np.arange(len(correlation_matrix.columns))+0.5, labels=benefits4_labels + ['Mean Salary'], rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(correlation_matrix.columns))+0.5, labels=benefits4_labels + ['Mean Salary'], rotation=0, va='center')
plt.figtext(0.5, -0.15, 'Correlation Matrix of Benefits Prevalence on Occupation-Year Level', wrap=True, horizontalalignment='right', fontsize=10)

## Median Salary

In [ ]:
# select all columns beginning with Prevalence: 
prevalence_cols = [col for col in df.columns if col.startswith('Prevalence') and col.endswith('(AI)')]
df_prevalence = df[prevalence_cols + ['MEDIAN_LOG_SALARY_ai']]

In [ ]:
df_prevalence

In [ ]:
plt.scatter(df['Prevalence: Paid Leave (AI)'], df['MEDIAN_LOG_SALARY_ai'])

In [ ]:
plt.scatter(df['Prevalence: Paid Leave (AI)'], df['Prevalence: Tuition Assistance (AI)'])

In [ ]:
remove_outliers(df, 'Prevalence: Paid Leave (AI)')[['Prevalence: Paid Leave (AI)','Prevalence: Tuition Assistance (AI)']].corr()

In [ ]:
plt.scatter(remove_outliers(df, 'Prevalence: Paid Leave (AI)')['Prevalence: Paid Leave (AI)'], remove_outliers(df,'Prevalence: Tuition Assistance (AI)')['Prevalence: Tuition Assistance (AI)'])

In [ ]:
correlation_matrix = df_prevalence.corr()

In [ ]:
correlation_matrix

In [ ]:
import seaborn as sns
plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, cmap='Blues', cbar = True)
plt.xticks(ticks=np.arange(len(correlation_matrix.columns))+0.5, labels=benefits4_labels + ['MEDIAN_LOG_SALARY_ai'], rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(correlation_matrix.columns))+0.5, labels=benefits4_labels + ['MEDIAN_LOG_SALARY_ai'], rotation=0, va='center')
plt.figtext(0.5, -0.15, 'Correlation Matrix of Benefits Prevalence on Occupation-Year Level', wrap=True, horizontalalignment='right', fontsize=10)